# Giá trị đơn Olist: lognormal hay gamma?

Đội tài chính muốn mô phỏng doanh thu và ước lượng tỉ lệ đơn lớn (trên 1 000 BRL) để đặt hạn mức
bảo hiểm hàng hoá. Bạn fit hai phân phối bằng MLE, so sánh bằng log-likelihood / AIC, KS, QQ-plot, và
kiểm tra xem phân phối "thắng" có mô tả đúng **phần đuôi** mà đội tài chính quan tâm không.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/05-econometrics-regression/exercises/intermediate/03-order-value-distribution -q`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=["order_delivered_customer_date"])

## 1. Giá trị đơn

Giá trị đơn = tổng `price` + tổng `freight_value` của mọi món trong đơn, chỉ cho đơn **giao xong**
(`order_status == "delivered"` và có `order_delivered_customer_date`).

In [ ]:
def order_values(items: pd.DataFrame, orders: pd.DataFrame) -> pd.Series:
    """Series index order_id, giá trị = tổng price + tổng freight_value; chỉ đơn giao xong."""
    delivered = orders.loc[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna(), "order_id"]
    totals = items.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1)
    return totals[totals.index.isin(delivered)].rename("order_value")

In [ ]:
# explore: hình dạng dữ liệu
import matplotlib.pyplot as plt

v = order_values(items, orders)
x = v.to_numpy()
print(len(x), v.describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_dict())
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(x[x < 1000], bins=100)
ax[0].set_title("Giá trị đơn (cắt ở 1 000 BRL)")
ax[1].hist(np.log(x), bins=100)
ax[1].set_title("log giá trị đơn")
plt.show()

## 2. MLE cho hai phân phối

- **Lognormal**: nghiệm đóng. $$\hat\mu$$, $$\hat\sigma$$ là trung bình và độ lệch chuẩn của
  $$\log x$$, chia cho $$n$$ (`ddof=0`), không phải $$n - 1$$.
- **Gamma** (shape, scale), `loc` cố định bằng 0: không có nghiệm đóng cho shape. Dùng
  `stats.gamma.fit(x, floc=0)` hoặc tự giải phương trình MLE.

In [ ]:
def fit_lognormal(x: np.ndarray) -> tuple[float, float]:
    """MLE (mu, sigma) của lognormal: mean và std (ddof=0) của log x."""
    lx = np.log(np.asarray(x, dtype=float))
    return float(lx.mean()), float(lx.std(ddof=0))


def fit_gamma(x: np.ndarray) -> tuple[float, float]:
    """MLE (shape, scale) của gamma với loc = 0."""
    shape, _, scale = stats.gamma.fit(np.asarray(x, dtype=float), floc=0)
    return float(shape), float(scale)

## 3. So sánh

`compare_fits(x, threshold)` trả DataFrame index `["lognorm", "gamma"]`, cột:
- `loglik`: tổng log mật độ tại tham số MLE.
- `aic`: $$2k - 2\log L$$ với $$k = 2$$.
- `ks_stat`: thống kê KS (khoảng cách lớn nhất giữa hai CDF), như `stats.kstest`.
- `p_exceed`: xác suất model gán cho $$X > \text{threshold}$$.

In [ ]:
def _frozen(x: np.ndarray) -> dict:
    mu, sigma = fit_lognormal(x)
    shape, scale = fit_gamma(x)
    return {"lognorm": stats.lognorm(s=sigma, scale=np.exp(mu)), "gamma": stats.gamma(a=shape, scale=scale)}


def compare_fits(x: np.ndarray, threshold: float = 1000.0) -> pd.DataFrame:
    """Index lognorm, gamma; cột loglik, aic, ks_stat, p_exceed (P(X > threshold) theo model)."""
    x = np.asarray(x, dtype=float)
    rows = {}
    for name, dist in _frozen(x).items():
        loglik = float(dist.logpdf(x).sum())
        rows[name] = {
            "loglik": loglik,
            "aic": 2 * 2 - 2 * loglik,
            "ks_stat": float(stats.kstest(x, dist.cdf).statistic),
            "p_exceed": float(dist.sf(threshold)),
        }
    return pd.DataFrame.from_dict(rows, orient="index")[["loglik", "aic", "ks_stat", "p_exceed"]]

In [ ]:
# explore: bảng so sánh và tỉ lệ thực tế vượt 1 000 BRL
table = compare_fits(x)
print(table)
print("thực tế P(X > 1000):", round((x > 1000).mean(), 4))

## 4. QQ-plot

`qq_points(x, dist, params)` trả `(theoretical, sample)`: `sample` là x đã sắp xếp tăng dần,
`theoretical` là quantile của phân phối tại vị trí $$p_i = (i - 0.5)/n$$, $$i = 1..n$$. `dist` là
`"lognorm"` (params = `(mu, sigma)`) hoặc `"gamma"` (params = `(shape, scale)`).

In [ ]:
def qq_points(x: np.ndarray, dist: str, params: tuple[float, float]) -> tuple[np.ndarray, np.ndarray]:
    """(quantile lý thuyết tại (i - 0.5)/n, x sắp xếp tăng dần). dist: lognorm hoặc gamma."""
    sample = np.sort(np.asarray(x, dtype=float))
    p = (np.arange(1, len(sample) + 1) - 0.5) / len(sample)
    if dist == "lognorm":
        mu, sigma = params
        theoretical = stats.lognorm.ppf(p, sigma, scale=np.exp(mu))
    elif dist == "gamma":
        shape, scale = params
        theoretical = stats.gamma.ppf(p, shape, scale=scale)
    else:
        raise ValueError(f"dist phải là lognorm hoặc gamma, nhận {dist!r}")
    return theoretical, sample

In [ ]:
# explore: QQ-plot trên thang log để thấy cả thân lẫn đuôi
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, (name, params) in zip(ax, [("lognorm", fit_lognormal(x)), ("gamma", fit_gamma(x))]):
    th, s = qq_points(x, name, params)
    a.loglog(th, s, ".", ms=2)
    a.loglog([s.min(), s.max()], [s.min(), s.max()], "k--", lw=0.8)
    a.set(title=f"QQ {name}", xlabel="quantile lý thuyết", ylabel="quantile mẫu")
plt.tight_layout()
plt.show()

In [ ]:
# explore: KS trên toàn bộ dữ liệu so với trên 500 đơn ngẫu nhiên
mu, sigma = fit_lognormal(x)
full = stats.kstest(x, "lognorm", args=(sigma, 0, np.exp(mu)))
sub = stats.kstest(np.random.default_rng(0).choice(x, 500, replace=False), "lognorm", args=(sigma, 0, np.exp(mu)))
print(f"n = {len(x)}: D = {full.statistic:.3f}, p = {full.pvalue:.1e}")
print(f"n = 500:   D = {sub.statistic:.3f}, p = {sub.pvalue:.2f}")

## 5. Khuyến nghị (bài mẫu)

Lognormal ($$\hat\mu = 4.691$$, $$\hat\sigma = 0.808$$) mô tả giá trị đơn tốt hơn gamma rõ rệt (AIC
thấp hơn khoảng 27 000, KS 0.026 so với 0.097), nên dùng nó để mô phỏng tổng doanh thu, vốn bị chi
phối bởi phần thân. **Không** dùng nó cho tỉ lệ đơn lớn: thực tế 1.15% đơn trên 1 000 BRL, lognormal
nói 0.30%, gamma 0.04%; QQ-plot cho thấy quantile mẫu vượt lên trên đường chéo từ khoảng
percentile 90 (quantile 99% của mẫu lớn hơn của lognormal 48%). KS p-value rất nhỏ trên 96 nghìn đơn chỉ nói dữ liệu không đúng tuyệt đối là lognormal
(trên 500 đơn ngẫu nhiên p = 0.78); nó không nói lognormal có đủ tốt cho việc của bạn không. Cho bảo
hiểm hàng hoá: dùng phân phối thực nghiệm cho phần đuôi (hoặc fit Pareto / generalized Pareto cho
phần trên percentile 95) ghép với lognormal cho phần thân, và kiểm tra lại theo từng danh mục sản phẩm.